## 0. 제출 정보
- 이름: 우다현
- GitHub ID: woodahyun
- 작성일: 2026-10-14
- 최종 제출 URL: https://github.com/woodahyun/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape:
[customers] (150, 6) / [products] (100, 4) / [orders] (300, 5) / [order_items] (764, 5)
- 결측치: 없음
- 전체 중복: 없음
- 주요 ID 중복: 없음
- 타입 문제 후보: [customers] `age`, `signup_date` / [products] `price` / [orders] `order_date` / [order_items] `quantity`, `unit_price`

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
실행 결과 네 파일 모두 결측치와 전체 행 중복이 없었고, 각 파일의 주요 ID도 중복되지 않았다. 다만, CSV를 읽은 초기 상태에서는 날짜와 숫자로 쓰일 컬럼이 문자열일 수 있으므로, 분석 전에 타입 변환이 필요하다.

### 나의 해석과 판단
먼저, 분석 금액과 기간에 직접 영향을 주는 `order_date`, `quantity`, `unit_price`, `price`의 타입과 범위를 확인해야 한다. 또한, 고객·주문·주문상세·상품을 연결하는 ID 관계를 점검해야 한다. 

### 업무·분석적 의미
전처리 전 상태를 남기면 행 수나 값이 바뀌었을 때 그 원인과 영향을 추적할 수 있다. 또한 원본 데이터를 덮어쓰지 않으면 처리 기준을 바꾸어도 처음 상태에서 다시 검증할 수 있게 된다.

### 한계와 추가 확인 사항
같은 고객이 여러 ID를 가졌는지, 주문 상태의 업무 정의가 무엇인지, 가격과 단가가 할인·쿠폰·배송비·세금을 반영하는지 등은 아직 확인하였다는 한계가 존재한다.

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 모든 문자열 컬럼의 앞뒤 공백을 제거하였다.
- 숫자 변환 대상과 실패 건수: `age`, `price`, `quantity`, `unit_price`를 `to_numeric(errors='coerce')`로 변환했으며 변환 실패로 새로 생긴 결측치는 0건이다.
- 날짜 변환 대상과 실패 건수: `order_date`, `signup_date`를 `to_datetime(errors='coerce')`로 변환했으며 실패 건수는 각각 0건이다.
- 결측 처리 기준: `age` 결측은 중앙값으로 대체하고 `city` 결측은 `Unknown`으로 구분하였다. 이번 데이터에서는 처리 전 결측이 없어 실제 값 변경은 없다.

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
문자열 공백을 제거하고 주문 상태를 대표 표기로 통일한 뒤에도 주문 상태는 `completed`, `cancelled`, `refunded`로 정상 분류되었다. 날짜 변환 실패와 숫자 변환 실패가 모두 0건이라, 이 샘플에서는 형식 변환 때문에 제외되거나 대체된 행이 없다.

### 나의 해석과 판단
결측이 없더라도 동일한 규칙을 코드에 남기는 것이 좋을 것 같다. 왜냐하면 새 데이터가 추가될 때 같은 규칙으로 처리할 수 있기 때문이다. 다만, `errors='coerce'`는 변환 실패 값을 결측으로 만들므로, 결측 대체나 행 제외 전에 실패 건수와 원문 값을 확인해야 할 것이다.

### 업무·분석적 의미
공백과 표기 차이를 정리하지 않으면 같은 주문 상태나 도시가 여러 그룹으로 나뉘어 집계될 수 있다. 날짜·숫자 타입을 명확히 하면 월별 분석과 금액 계산을 재현 가능하게 수행할 수 있다.

### 한계와 추가 확인 사항
현재 날짜 변환이 성공하였더라도 주문일 자체가 실제 주문 시점과 일치한다는 보장은 아직 없는 상태이다. 연령 중앙값 대체와 도시 `Unknown` 처리도 결측 발생 원인에 따라 분석 편향을 만들 수 있으므로, 운영 데이터에서는 결측 사유를 별도로 확인하는 과정이 추가로 필요하다.


## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 0건
- 표준화한 범주값: `order_status`를 `completed`, `cancelled`, `refunded`의 대표 표기로 통일하도록 처리하였다. 
- 허용값 밖 값: `price <= 0`, `quantity <= 0`, `unit_price <= 0`은 각각 0건이다.
- 이상값 후보: 확인되지 않음.

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
이 데이터에는 삭제할 중복 행과 0 이하의 가격·수량·단가가 없었다. 따라서 이상값 처리 기준을 적용한 뒤에도 `products`는 100행, `order_items`는 764행으로 유지되었다.

### 나의 해석과 판단
이상값 후보를 발견해도 바로 삭제하면 안되고, 삭제 전 값의 입력 오류 여부와 업무 의미를 확인한 뒤 분석 목적에 맞는 포함·제외 기준을 정하는 과정이 필요하다.

### 업무·분석적 의미
범주 표준화는 그룹별 집계의 정확성을 높이고, 이상값 기준을 문서화하면 같은 분석을 다시 실행할 때 결과 차이를 설명할 수 있다.

### 한계와 추가 확인 사항
현재 데이터에는 할인, 반품 사유, 행사 여부가 없어 높은 단가나 특정 수량이 정상인지 완전히 판단할 수 없다는 한계가 존재한다. 또한 주문 상태 표준화 규칙은 실제 운영 시스템의 전체 상태 코드 목록으로 검증이 필요하다.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0건 (`orders.customer_id → customers.customer_id`)
- 없는 `order_id`: 0건 (`order_items.order_id → orders.order_id`)
- 없는 `product_id`: 0건 (`order_items.product_id → products.product_id`)
- 생성한 파생 컬럼: `orders_clean`의 `order_month`, `order_dayofweek`, `order_items_clean`의 `line_total`
- `line_total` 검증: `quantity * unit_price` > 전처리 후 주문 상세 금액 합계: 255,610,000원

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
전처리 뒤에도 세 파일 간 참조되지 않는 ID는 전부 0건이었다. `line_total`은 수량과 단가가 모두 양수인 주문 상세를 기준으로 계산되었고, 주문 상세 데이터에 1개 컬럼이 추가되었다.

### 나의 해석과 판단
전처리 과정에서 행을 제거하면 참조 관계가 깨질 수 있으므로, 전처리 전 검증만으로는 충분하지 않다. 이 데이터의 경우 제거된 행이 없어 관계가 유지되었지만, 향후 상품이나 주문 상세를 제외하는 기준이 바뀌면 다시 검증이 필요하다.

## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| customers 행 수 | 150행 | 150행 | 결측·중복·이상값이 없어 행 수 변화 없음 |
| products 행 수 | 100행 | 100행 | 0 이하 가격이 없어 제외 행 없음 |
| orders 행 수 | 300행 | 300행 | 행 수 변화는 없으나 `order_month`, `order_dayofweek` 추가되어 5열 > 7열로 변화 |
| order_items 행 수 | 764행 | 764행 | 행 수 변화는 없으나 `line_total` 추가되어 5열 > 6열로 변화 |
| 결측 | 모두 0건 | 모두 0건 | 점검 결과 유지 |
| 중복 | 모두 0건 | 모두 0건 | 점검 결과 유지 |
| 변환 실패 | 모두 0건 | 모두 0건 | 점검 결과 유지 |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
행 수는 4가지 dataset에서 모두 일정하였다. orders 에서는 2개, order_items 에서는 1개의 열이 추가되었다. 

### 나의 해석과 판단
전처리 전후 비교를 통해 실제로 무엇이 바뀌었는지 확인할 수 있다. 이번에는 행 수 감소가 없었지만, 이후 데이터에서 감소가 발생한다면 어떤 필터가 몇 행을 제외했는지 반드시 기록해야 할 것이다.

### 업무·분석적 의미
비교표는 분석 결과 차이가 데이터 변화 때문인지, 집계 로직 변화 때문인지 구분하는 근거가 되므로, 다른 사람과의 협업 시에도 다른 사람이 처리 범위를 빠르게 이해할 수 있도록 해준다.

### 한계와 추가 확인 사항
현재 비교는 행 수·결측·중복·변환 실패만 보는 것이므로, 값 자체가 잘못 입력되었는지, 고객 ID가 실제 한 고객을 유일하게 대표하는지 등의 품질에 대한 추가 검증이 필요하다.

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: `customers_clean.csv`, `products_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`
- 생성된 요약 보고서: `reports/ch05_preprocessing_summary.md`
- `python scripts/preprocess_data.py` 재실행 결과: 성공
- 재실행 후 달라진 점: 없음

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
otebook에서 한 번 실행되는 것과 반복 실행 가능한 분석은 다르다. Notebook은 중간 셀 실행 순서나 메모리 상태에 영향을 받을 수 있으므로, 핵심 전처리 로직을 함수와 스크립트로 분리해 처음부터 다시 실행해 보는 과정이 필요할 것이다. 재현 가능한 스크립트와 보고서가 있으면 새 파일이 들어왔을 때 같은 품질 점검을 반복할 수 있고, 다른 사람이 같은 결과를 검증하기 쉬워질 것이다.


## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본 데이터는 보존하며, 전처리 결과는 별도 파일로 저장한다.
2. 결측·중복·타입·범주·이상값은 바로 삭제하지 않고, 데이터 의미와 분석 목적을 기준으로 삭제 여부를 판단한 후 처리한다.
3. 처리 기준·행 수 변화·관계 검증 결과를 기록하고 스크립트로 재실행 가능하게 만든다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
가장 위험한 자동 처리는 변환 실패값이나 이상값을 자동적으로 삭제하도록 처리하는 것이다. 특정 고객·주문이 체계적으로 빠져 분석 결과를 왜곡할 수 있고, 나중에 누락 사실을 발견하기도 어렵기 때문이다.

### 다음 EDA에서 특히 주의할 데이터 특성
주문 상태를 구분해야 한다. `completed`만 매출 분석에 포함할지, `cancelled`와 `refunded`를 어떻게 처리할지는 분석 질문에 따라 명확한 정의가 필요하다. 또한 `line_total`은 수량×단가이므로, 순매출이나 실제 결제액과 동일하다고 가정하면 안 된다.

### 현재 전처리의 한계
할인·쿠폰·배송비·세금·반품 사유·실결제 금액이 없어 순매출과 고객의 실제 지출을 정확히 계산하는 데 한계가 존재한다. 샘플 기간도 주문일 기준 2025-07-09부터 2026-07-08까지이므로 월별 비교 시 시작·종료 월은 전체 월이 아닐 수 있다는 점을 주의해야 한다.

## 최종 제출 체크
- [o] raw 원본을 덮어쓰지 않았습니다.
- [o] 처리 전/후 비교를 남겼습니다.
- [o] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [o] clean CSV와 재실행 결과를 확인했습니다.
- [o] 개인정보/Secret이 없습니다.
- [o] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [o] 최종 Notebook 파일 URL을 제출합니다.